# Metabolomics Data Preprocessing

This notebook preprocesses metabolomics data from MetaboLights studies for feature selection analysis.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display, Markdown

# Set display options
pd.set_option("display.precision", 3)


## 1. Choose Dataset

Available datasets:
- **MTBLS1**: Type 2 Diabetes (132 samples, 220 features)
- **MTBLS12968**: Polycystic Ovarian Syndrome - PCOS (149 samples, 486 features)
- **MTBLS2**: Arabidopsis thaliana plant study (16 samples, 15,272 features)

In [ ]:
# Choose which dataset to use
dataset_name = "pcos"

In [ ]:
name2id = {
    "diabetes": "MTBLS1",
    "pcos": "MTBLS12968",  #  Y["control_vs_patient"] = Y["Factor Value[Group]"].str.split("-", n=1).str[0]
    "arabidopsis": "MTBLS2",
    "yeast": "MTBLS8",  # 126 samples × 383 metabolites ...cell growth control, not sure what the classes are (6?)
    # https://www.ebi.ac.uk/metabolights/editor/MTBLS8/descriptors
}
study_id = name2id[dataset_name]

# Create file path
data_dir = Path("../data/metabolights")
file_path = data_dir / f"metabolights_{study_id}.csv"

display(Markdown(f"**Selected dataset:** {dataset_name} ({study_id})"))
display(Markdown(f"**File path:** `{file_path}`"))
display(Markdown(f"**File exists:** {file_path.exists()}"))

## 2. Load data

In [ ]:
# Load the CSV file
df_raw = pd.read_csv(file_path, index_col=0)

metadata = df_raw[[col for col in df_raw.columns if col.startswith("metadata")]]
measurements = df_raw.drop(columns=metadata.columns)
identified_features = [
    col for col in measurements.columns if not col.startswith("unidentified")
]

display(Markdown("✓ Dataset loaded successfully!"))
display(Markdown("**Shape:**"))
display(Markdown(f"- {measurements.shape[0]} samples"))
display(Markdown(f"- {len(identified_features)} identified features"))
display(
    Markdown(
        f"- {len(measurements.columns) - len(identified_features)} unidentified features"
    )
)
display(Markdown(f"- {len(metadata.columns)} metadata columns"))

# display(
#     Markdown(
#         f"**Memory usage:** {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
#     )
# )

## 3. Display Data Overviews

### 3.1 Metadata

In [ ]:
display(Markdown("**Metadata columns:**"))
for col in metadata.columns:
    display(Markdown(f"- {col}"))

display(Markdown("**Value counts:**"))
display(metadata.value_counts())


### 3.2 Measurements

In [ ]:
# Display first few rows
display(Markdown("**Sampled measurements:**"))
display(measurements.sample(10))

## 4. Define response and stratification

In [ ]:
# map the study ID to response variable name to be extracted from Y
id2response = {
    "MTBLS1": "metadata__Factor Value[Metabolic syndrome]",
    "MTBLS12968": "metadata__PCOS",
    # "MTBLS12968": "metadata__PRETERM",
    "MTBLS2": "metadata__Factor Value[Genotype]",
}

# define which stratifiers are not response variables
# and thus should be 1-hot encoded
id2onehot = {
    "MTBLS1": ["metadata__Factor Value[Gender]"],
    "MTBLS12968": ["metadata__PRETERM"],
    # "MTBLS12968": ["metadata__PCOS"],
}

# define which stratifiers to use for stratified sampling
# if the stratification vector is only the response, do not include it here
# if there are more, include all INCLUDING the response variable
id2stratifiers = {
    "MTBLS1": [
        "metadata__Factor Value[Metabolic syndrome]",
        "metadata__Factor Value[Gender]",
    ],
    "MTBLS12968": [
        "metadata__PCOS",
        "metadata__PRETERM",
    ],
    "MTBLS2": [
        "metadata__Factor Value[Genotype]",
        "metadata__Factor Value[Replicate]",
    ],
}


# (binary) values from the response variable
# control is always 0
id2binary = {
    "MTBLS1": {
        "Control Group": 0,
        "diabetes mellitus": 1,
    },
    "MTBLS12968": {
        "pcos": 1,  # PCOS diagnosis
        "control": 0,
        "preterm": 1,  # preterm delivery
        "term": 0,
    },
    "MTBLS2": {
        "Col-0": 0,
        "cyp79": 1,
    },
}


## 5. Construct final dataset


- Get a binary response variable
- Fill missing values with 0 (assuming a given metabolite was not found in the sample)
- Optionally, drop unidentified metabolites
- Optionally, define a stratification vector

In [ ]:
if dataset_name == "arabidopsis":
    filter_only_identified = True
    df = measurements[identified_features]
else:
    filter_only_identified = False
    df = measurements

In [ ]:
# Define stratification groups if given for this study
# aggregate stratifiers into a single column
is_stratified = False
if study_id in id2stratifiers:
    stratifiers = id2stratifiers[study_id]
    df["stratification_groups"] = metadata[stratifiers].agg("_".join, axis=1)
    # turn the stratification groups into integers
    df["stratification_groups"] = (
        df["stratification_groups"].astype("category").cat.codes
    )
    is_stratified = True

In [ ]:
# One-hot encode non-response stratifiers if given for this study
# and concatenate to the main dataframe
# will be treated as regular features
if study_id in id2onehot:
    onehot_candidates = id2onehot[study_id]
    onehots = pd.get_dummies(metadata[onehot_candidates], prefix_sep="=")
    # do not include the encoded metadata columns as features, only the one-hot encoded columns
    onehots = onehots.drop(columns=onehot_candidates, errors="ignore")
    # convert the one-hot encoded columns to integers (0/1)
    onehots = onehots.astype(int)
    df = pd.concat([df, onehots], axis=1)

In [ ]:
# Define response variable
df["response"] = metadata[id2response[study_id]].map(id2binary[study_id])

# Fill missing values with 0 (assuming missing means not detected)
df = df.fillna(0)

df.info()

## 6. Save the dataset

In [ ]:
# drop the 'metadata__" prefix from the response variable name for better readability in the output file name
response_name = id2response[study_id].split("metadata__")[-1]

if is_stratified:
    real_p = (
        df.shape[1] - 2
    )  # exclude response variable and stratification group from feature count
else:
    real_p = df.shape[1] - 1  # exclude response variable from feature count

# save to csv
output_file_path = (
    data_dir
    / f"{dataset_name}_{study_id}_preprocessed_n={df.shape[0]}_p={real_p}_response={response_name}_onlyidentified={filter_only_identified}_stratified={is_stratified}.csv"
)
df.to_csv(output_file_path)
display(Markdown(f"✓ Preprocessed dataset saved to `{output_file_path}`"))

## 7. Explore final dataset

In [ ]:
# Final dataset information
display(Markdown("**Final Dataset Info:**"))
df.info()

In [ ]:
# Statistical summary
display(Markdown("**Statistical Summary:**"))
display(df.describe())

In [ ]:
# Visualize feature distributions in a grid of subplots
display(Markdown("**Feature distributions:**"))

# Get feature columns (exclude response and any stratification columns)
feature_cols = [
    col for col in df.columns if col not in ["response", "stratification_groups"]
]
n_features = len(feature_cols)

# Configure grid layout
cols_per_row = 6
n_rows = (n_features + cols_per_row - 1) // cols_per_row

# Create subplots with better spacing
fig = make_subplots(
    rows=n_rows,
    cols=cols_per_row,
    subplot_titles=feature_cols,
    vertical_spacing=0.008,
    horizontal_spacing=0.02,
)

# Add histograms
for i, col in enumerate(feature_cols):
    row = i // cols_per_row + 1
    col_idx = i % cols_per_row + 1

    fig.add_trace(
        go.Histogram(x=df[col], nbinsx=20, showlegend=False, marker_color="steelblue"),
        row=row,
        col=col_idx,
    )

# Update layout
fig.update_layout(
    height=200 * n_rows,
    width=1800,
    title_text="Feature Distributions",
    showlegend=False,
)

# Update axes
fig.update_xaxes(showticklabels=True, title_text="")
fig.update_yaxes(showticklabels=True, title_text="")

# Make subplot titles smaller
fig.update_annotations(font_size=8)

fig.show()

In [ ]:
# show correlation heatmap
display(Markdown("**Feature correlation heatmap:**"))
corr_matrix = df[feature_cols].corr()
fig_corr = px.imshow(
    corr_matrix,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Feature Correlation Heatmap",
)
fig_corr.update_layout(height=1400, width=1600)
fig_corr.show()

In [ ]:
# Check response variable distribution
display(Markdown("**Response variable distribution:**"))
display(df["response"].value_counts())
display(df["response"].value_counts(normalize=True).round(2))


In [ ]:
# visualize in pie plot: stratification groups distribution
if "stratification_groups" in df.columns:
    group_counts = df["stratification_groups"].value_counts()
    fig = go.Figure(
        data=[
            go.Pie(
                labels=group_counts.index,
                values=group_counts.values,
                hole=0.4,
                textinfo="label+percent",
            )
        ]
    )
    fig.update_layout(title="Stratification Groups Distribution", width=600)
    fig.show()

In [ ]:
# Show correlations of features with response variable and stratification groups (if exists)
display(
    Markdown(
        "**Correlation of features with response variable and stratification groups:**"
    )
)
feature_cols = [
    col for col in df.columns if col not in ["response", "stratification_groups"]
]

# Calculate correlations
if "stratification_groups" in df.columns:
    # Encode stratification groups numerically
    df_temp = df.copy()
    df_temp["stratification_groups_encoded"] = pd.Categorical(
        df_temp["stratification_groups"]
    ).codes

    # Calculate correlation with response
    corr_response = df_temp[feature_cols].corrwith(df_temp["response"])

    # Calculate correlation with stratification groups
    corr_strat = df_temp[feature_cols].corrwith(
        df_temp["stratification_groups_encoded"]
    )

    # Combine into a dataframe
    corr_with = pd.DataFrame(
        {"response": corr_response, "stratification_groups": corr_strat}
    ).T
else:
    corr_with = df[feature_cols].corrwith(df["response"]).to_frame(name="response").T

# Create heatmap
fig_corr_with = px.imshow(
    corr_with,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation of Features with Response and Stratification Groups"
    if "stratification_groups" in df.columns
    else "Correlation of Features with Response Variable",
    aspect="auto",
)
fig_corr_with.update_layout(height=300, width=2000)
fig_corr_with.update_xaxes(tickangle=90, tickfont=dict(size=8))
fig_corr_with.show()